# Things to analise
[x] Resulting band ids


[x] Join with labels

[x] Join with releases

[x] Join with countries

[x] Join with genres

[x] Data exploration (top 5 of every exploration)
    - Bands from countries
    - Bands with same label
    - Bands with same genre

In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve().parent

os.chdir(PROJECT_ROOT)

working_dir = os.getcwd()
print(working_dir)

C:\Users\Ocean\Desktop\Semestre 12 - TFG\ontometal


# Band data checks

The idea is to check generated data for normalized_v2 dataset creation with correspondant corrections.

In [3]:
bands = pd.read_csv("src/etl/data/normalized/bands.csv").copy()

C:\Users\Ocean\AppData\Local\Temp\ipykernel_10636\1933402691.py:1: DtypeWarning: Columns (0: bandId) have mixed types. Specify dtype option on import or set low_memory=False.
  bands = pd.read_csv("src/etl/data/normalized/bands.csv").copy()


In [4]:
# Number of bands
bands.shape

(183209, 8)

In [5]:
# Basic info about the bands dataset
bands.info()

<class 'pandas.DataFrame'>
RangeIndex: 183209 entries, 0 to 183208
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   bandId           183209 non-null  object 
 1   bandName         183204 non-null  str    
 2   status           183209 non-null  str    
 3   metalArchiveUrl  183209 non-null  str    
 4   releases         183209 non-null  str    
 5   producedBy       31240 non-null   float64
 6   hasGenre         183209 non-null  str    
 7   hasCountry       183209 non-null  int64  
dtypes: float64(1), int64(1), object(1), str(5)
memory usage: 11.2+ MB


In [6]:
# Bands with Na values
bands.isna().sum().sort_values(ascending=False)

producedBy         151969
bandName                5
status                  0
bandId                  0
metalArchiveUrl         0
releases                0
hasGenre                0
hasCountry              0
dtype: int64

In [7]:
# Number of duplicated
bands.duplicated().sum()

np.int64(68)

In [8]:
# Numnber of unique bandIds and bandNames
print(bands["bandId"].nunique())
print(bands["bandName"].nunique())

182565
152009


In [9]:
# Number of bands with same bandId
bands.groupby("bandId").size().reset_index(name="count").query("count > 1").shape[0]

283

In [10]:
# Ids duplicated
bands.groupby("bandId").size().reset_index(name="count").where(
    lambda x: x["count"] > 1
).dropna()

,bandId,count
0,0,10.0
1,1,15.0
2,2,19.0
3,3,45.0
4,4,25.0
...,...,...
179426,3540547326,2.0
179846,3540548155,2.0
180294,3540548969,2.0
180428,3540549258,2.0


Need to remove duplicated id bands for data consistency and no name bands

# Bands join countries

In [13]:
bands = pd.read_csv("src/etl/data/normalized/bands.csv", low_memory=False).copy()
countries = pd.read_csv("src/etl/data/normalized/countries.csv").copy()
bands.merge(countries, left_on="hasCountry", right_on="id", how="left").drop(
    columns=["id"]
).rename(columns={"name": "countryName"})[["bandId", "countryName"]]

,bandId,countryName
0,3540442600,united states
1,3540525193,colombia
2,3540473101,united kingdom
3,3540535978,united states
4,3540352307,united states
...,...,...
183204,97695,"korea, south"
183205,3540435197,"korea, south"
183206,3540473622,japan
183207,3540493653,japan


In [14]:
# All bands has a country assigned
bands["hasCountry"].isna().sum()

np.int64(0)

# Bands join genres
We already know not all bands will have genres

In [16]:
# We merge with genre name because we ensure on etl to map genres
# Lets ensure genres are mapped correctly and if not, theres no strage values in genres
import ast
bands = pd.read_csv("src/etl/data/normalized/bands.csv", low_memory=False).copy()
genres = pd.read_csv("src/etl/data/normalized/genres.csv").copy()
genres["name"] = genres["name"].astype(str).str.strip().str.lower()

bands["hasGenre"] = bands["hasGenre"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

used = set(
    bands["hasGenre"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)


unused_genres = genres[~genres["name"].isin(used)]

print("Total genres:", len(genres))
print("Used genres:", len(used))
print("Unused genres:", len(unused_genres))

Total genres: 1375
Used genres: 1288
Unused genres: 87


In [17]:
unused_genres.head()

,id,name
5,6,brutal death metal with industrial influences
17,18,crust n roll
22,23,industrial gothic rock
37,38,experimental post-hardcore
50,51,grind n roll


In [18]:
pd.set_option("display.max_colwidth", None)
# Check why are not used against raw (maybe because the cleaning process)
raw_bands = pd.read_csv("src/etl/data/raw/metal_bands_roster.csv").copy()
raw_bands[
    raw_bands["Genre"].str.contains("brutal death metal with industrial influences", case=False, na=False)
][["Genre"]]

C:\Users\Ocean\AppData\Local\Temp\ipykernel_10636\2216281253.py:3: DtypeWarning: Columns (0: Band ID) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_bands = pd.read_csv("src/etl/data/raw/metal_bands_roster.csv").copy()


,Genre
142071,Groove/Nu-Metal (early); Brutal Death Metal with Industrial influences (later)


Cleaning process not the same for genres, labels and genres

# Bands join labels

There are duplicated values on labels raw dataset, let's check that the resulting label corresponds to the value assinged on complete_roser dataset

In [19]:
# Check labels assigned to bands correspond to labels in raw labels dataset
complete_roster = pd.read_csv("src/etl/data/raw/complete_roster.csv").copy()
raw_labels = pd.read_csv("src/etl/data/raw/labels_roster.csv").copy()
labels = pd.read_csv("src/etl/data/normalized/labels.csv").copy()

C:\Users\Ocean\AppData\Local\Temp\ipykernel_10636\1618164537.py:2: DtypeWarning: Columns (0: Band ID, 1: Name_y, 2: Specialization, 3: Status_y, 4: Country_y, 5: Website, 6: Online Shopping) have mixed types. Specify dtype option on import or set low_memory=False.
  complete_roster = pd.read_csv("src/etl/data/raw/complete_roster.csv").copy()


In [20]:
# Check raw label from first label on normalized labels
import ast

label_name = labels["labelName"].iloc[0]
print("Label name:", label_name)

label_id = labels["labelId"].iloc[0]
print("Label ID:", label_id)

band_produced = ast.literal_eval(labels["producer"].iloc[0])
band_produced = list(map(int, band_produced))[0]

print("Band produced:", band_produced)

Label name: nuclear blast
Label ID: 2
Band produced: 213


In [21]:
complete_roster.columns
complete_roster[
    (complete_roster["Label ID"] == label_id) &
    (complete_roster["Band ID"] == band_produced)
]

,Band ID,Name_x,URL,Country_x,Genre,Status_x,Photo_URL,Label ID,Name_y,Specialization,Status_y,Country_y,Website,Online Shopping
1261,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,Nuclear Blast,"Hardcore (early), Metal and subgenres",active,Germany,http://www.nuclearblast.de/en/shop/index.html,Yes
1262,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2 Headed Baby Productions,Buzzov•en,closed,United States,NaN,NaN
1263,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2 Loud 4 U Music Publishing,NaN,closed,Germany,NaN,NaN
1264,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2 Metri Sottoterra,NaN,unknown,Italy,NaN,NaN
1265,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2 Slow Records,NaN,active,Thailand,http://2slow.tk/,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1697,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2R Records,NaN,closed,NaN,NaN,NaN
1698,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2S Productions,NaN,unknown,Russia,NaN,NaN
1699,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2TDF,NaN,closed,Austria,NaN,NaN
1700,213,Aborted,https://www.metal-archives.com/bands/Aborted/213,Belgium,Death Metal,Active,https://www.metal-archives.com/images/2/1/3/213_photo.jpg?3507,2.0,2XL Records,NaN,closed,Germany,NaN,NaN


Since there are in some cases multiple labels for one band, we must select the most complete data from labels

- An active label and most complete columns 

# Join with releases

In [22]:
releases = pd.read_csv("src/etl/data/normalized/releases.csv").copy()
import ast

bands["releases"] = bands["releases"].apply(ast.literal_eval)

In [23]:

# Correct join
bands = bands.explode("releases")

merged = bands.merge(
    releases,
    left_on="releases",
    right_on="releasedBy",
    how="left"
)

merged

,bandId,bandName,status,metalArchiveUrl,releases,producedBy,hasGenre,hasCountry,releaseId,releaseTitle,releaseYear,releaseType,releasedBy
0,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solution/3540442600,1,NaN,"[crust punk with doom metal influences, thrash metal]",1,23721.0,disment of soul,1991,demo,1.0
1,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solution/3540442600,1,NaN,"[crust punk with doom metal influences, thrash metal]",1,23722.0,disment of soul,1991,demo,1.0
2,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solution/3540442600,1,NaN,"[crust punk with doom metal influences, thrash metal]",1,23723.0,disment of soul,1991,demo,1.0
3,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solution/3540442600,1,NaN,"[crust punk with doom metal influences, thrash metal]",1,23724.0,disment of soul,1991,demo,1.0
4,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solution/3540442600,1,NaN,"[crust punk with doom metal influences, thrash metal]",1,23725.0,disment of soul,1991,demo,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1251403,3540493653,鬼嫁,split-up,https://www.metal-archives.com/bands/%e9%ac%bc%e5%ab%81/3540493653,651225,NaN,"[heavy metal, heavy metal with hard rock]",26,NaN,NaN,NaN,NaN,NaN
1251404,3540493653,鬼嫁,split-up,https://www.metal-archives.com/bands/%e9%ac%bc%e5%ab%81/3540493653,651226,NaN,"[heavy metal, heavy metal with hard rock]",26,NaN,NaN,NaN,NaN,NaN
1251405,37240,魚雷,split-up,https://www.metal-archives.com/bands/%e9%ad%9a%e9%9b%b7/37240,651227,NaN,[industrial thrash metal],26,NaN,NaN,NaN,NaN,NaN
1251406,37240,魚雷,split-up,https://www.metal-archives.com/bands/%e9%ad%9a%e9%9b%b7/37240,651228,NaN,[industrial thrash metal],26,NaN,NaN,NaN,NaN,NaN


# Data exploration on bands

In [24]:
bands_exp = pd.read_csv("src/etl/data/normalized/bands.csv").copy()
countries = pd.read_csv("src/etl/data/normalized/countries.csv").copy()
labels = pd.read_csv("src/etl/data/normalized/labels.csv").copy()
genres = pd.read_csv("src/etl/data/normalized/genres.csv").copy()

C:\Users\Ocean\AppData\Local\Temp\ipykernel_10636\1753534331.py:1: DtypeWarning: Columns (0: bandId) have mixed types. Specify dtype option on import or set low_memory=False.
  bands_exp = pd.read_csv("src/etl/data/normalized/bands.csv").copy()


## Top 5 countries by number of bands

In [25]:
top_countries = (
    bands_exp.merge(countries, left_on="hasCountry", right_on="id", how="left")
    .groupby("name")
    .size()
    .reset_index(name="bandCount")
    .sort_values("bandCount", ascending=False)
    .head(5)
)
top_countries

,name,bandCount
149,united states,41374
49,germany,13834
20,brazil,8392
70,italy,8099
148,united kingdom,7509


## Top 5 labels by number of bands produced

In [26]:
import ast

labels["_bandCount"] = labels["producer"].apply(
    lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0
)

top_labels = (
    labels[["labelId", "labelName", "_bandCount"]]
    .sort_values("_bandCount", ascending=False)
    .head(5)
)
top_labels.drop(columns=["_bandCount"]).assign(bandCount=top_labels["_bandCount"])

,labelId,labelName,bandCount
1,3,metal blade records,100
4,6,massacre records,100
59,76,avantgarde music,100
0,2,nuclear blast,100
67,86,coyote records,100


## Top 5 genres by number of bands

In [27]:
bands_exp["hasGenre"] = bands_exp["hasGenre"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

top_genres = (
    bands_exp["hasGenre"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
    .head(5)
    .reset_index()
    .rename(columns={"index": "genre", "hasGenre": "genre", "count": "bandCount"})
)
top_genres

,genre,bandCount
0,death metal,38732
1,black metal,36240
2,thrash metal,32229
3,heavy metal,23814
4,doom metal,12163


# Summary of changes needed v1

1. 283 bands have same id, well delete those bands to keep data consistent. If all are added we will not know which band the process is picking every time

2. 5 bands have null Name, delete it

3. 87 genres are not used, it may be caused due to cleaning process. Check that process and may need to join through genre id better than name. Example cleaning -> grin n' roll = grid n roll (not matching on name)

4.  Get most complete data from labels since there are duplicated label ids but all match on complete roaster data.

# Decisions made during development process

- Labels has specializations that not match a metal genre, like `Jason Newsted's` which means that the label only procuces music from bassis Jason Newsted. Due to this we do not use specializations from labels to normalize genres

- Used a map of genres to map on genre name but cleansing is making 87 genres to not match so clieansing must be changed.

- Labels collect all produced bands

# Notes on data types

- `ast.literal_eval` is needed when reading list columns (e.g. `releases`, `hasGenre`, `producer`, `hasSpecialization`) because CSV does not support list types natively. `to_csv()` serializes Python lists as their `str()` representation (e.g. `"[1, 2, 3]"`), and `read_csv` reads them back as plain strings.

- `Int64` (nullable integer) is used for foreign key columns (`producedBy`, `hasCountry`) because left joins introduce `NaN` values, and regular `int64` cannot hold `NaN`.